In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
import random

## 1. 加载数据

In [ ]:
data = loadmat('cluster_dataset.mat')
X = data['data']
print(f"数据集大小: {X.shape}")

# 先看看数据长什么样
x1 = X[:, 0]
x2 = X[:, 1]
plt.figure(dpi=150)
plt.scatter(x1, x2, color='b', alpha=0.8)
plt.xlabel('x1')
plt.ylabel('x2')
plt.title('原始数据')
plt.show()

# 肉眼大概能看出3簇？

## 2. K-Means算法

K-Means步骤：
1. 随机选k个中心
2. 把每个点分到最近的中心
3. 重新算每个簇的中心（取平均）
4. 重复2-3直到中心不再变

代价函数：所有点到其中心的距离平方和
$$J = \sum_{k=1}^{K} \sum_{x_i \in C_k} ||x_i - \mu_k||^2$$

In [ ]:
def init_centers(X, k):
    """
    随机选k个点作为初始中心
    
    运作流程：
        1. 从数据集中随机选k个不重复的索引
        2. 把这些点取出来作为中心
    
    重要变量：
        m: 样本总数（局部变量）
        indices: 随机选的索引（局部变量）
        centers: 初始中心（局部变量，返回值）
    """
    m = X.shape[0]
    indices = random.sample(range(m), k)
    centers = []
    for i in indices:
        centers.append(X[i].copy())
    return centers

In [ ]:
def distance(v1, v2):
    """
    算两个向量的欧氏距离
    d = sqrt(sum((v1_i - v2_i)^2))

    运作流程：
        1. 算两个向量对应元素的差
        2. 差值平方后求和
        3. 开根号得到距离

    重要变量：
        v1, v2: 两个向量

    依赖关系：
        - 依赖numpy的求和和开方运算
    """
    return np.sqrt(np.sum((v1 - v2) ** 2))

In [ ]:
def cluster_assignment(X, centers):
    """
    把每个点分到最近的中心
    
    运作流程：
        1. 对每个点，算它到所有中心的距离
        2. 找最近的中心，把这个点分过去
    
    重要变量：
        assignment: 字典，key是簇编号，value是属于这个簇的点（局部变量）
        dists: 到各中心的距离列表（局部变量）
        nearest: 最近的中心编号（局部变量）
    
    依赖关系：依赖distance()算距离
    """
    assignment = {}
    for i in range(len(centers)):
        assignment[i] = []
    for i in range(X.shape[0]):
        dists = []
        for c in range(len(centers)):
            dists.append(distance(X[i], centers[c]))
        nearest = np.argmin(dists)
        assignment[nearest].append(X[i])
    return assignment

In [ ]:
def cost_function(assignment, centers):
    """
    算代价函数：所有点到其中心的距离平方和
    就是sklearn里的inertia（这个名词也是问AI才知道的）
    
    运作流程：
        1. 对每个簇的每个点，算到中心的距离平方
        2. 全部加起来
    
    重要变量：
        total: 距离平方和（局部变量）
    
    依赖关系：依赖distance()算距离
    """
    total = 0
    for k in assignment:
        for point in assignment[k]:
            total = total + distance(point, centers[k]) ** 2
    return total

In [ ]:
def center_update(assignment, centers):
    """
    更新中心：每个簇的中心变成这个簇所有点的平均值
    
    运作流程：
        1. 对每个簇，算里面所有点的平均
        2. 如果簇是空的，中心不变
           （这种情况应该很少出现，但万一出现了不能报错）
        3. 看中心有没有变，没变就停机
    
    重要变量：
        new_centers: 新中心（局部变量）
        stop: 1表示停机，0表示继续（局部变量）
    """
    new_centers = []
    stop = 1
    for k in range(len(centers)):
        cluster = np.array(assignment[k])
        if len(cluster) > 0:
            new_center = np.mean(cluster, axis=0)
        else:
            # 空簇，中心不变
            new_center = centers[k].copy()
        new_centers.append(new_center)
        if not np.array_equal(new_center, centers[k]):
            stop = 0
    return new_centers, stop

In [ ]:
def plot_clustering(assignment, centers, epoch):
    """
    画聚类结果

    运作流程：
        1. 创建画布
        2. 对每个簇，用不同颜色画散点
        3. 用黑色星号标记中心点
        4. 加上坐标轴标签和标题

    重要变量：
        - color: 颜色列表，每个簇一种颜色
        - cluster: 某个簇的所有点

    依赖关系：
        - 依赖assignment和centers（来自kmeans函数）
        - 依赖matplotlib.pyplot
    """
    color = ['r', 'b', 'c', 'g', 'k', 'w', 'y', 'm']
    plt.figure(dpi=150)
    for k in range(len(centers)):
        cluster = np.array(assignment[k])
        if len(cluster) == 0:
            continue
        plt.scatter(cluster[:, 0], cluster[:, 1], c=color[k])
    for k in range(len(centers)):
        plt.scatter(centers[k][0], centers[k][1], c='k', marker='*', s=100)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.title('epoch ' + str(epoch))
    plt.show()

In [ ]:
def kmeans(X, k, max_epoch, plot=True):
    """
    K-Means主函数
    
    运作流程：
        1. 随机初始化中心
        2. 循环：分配→更新中心→画图→判断停机
        3. 算最终代价
    
    重要变量：
        centers: 中心点（局部变量）
        assignment: 分配结果（局部变量）
        stop: 停机标识（局部变量）
        cost: 代价函数值（局部变量）
    
    依赖关系：
        - init_centers() 初始化
        - cluster_assignment() 分配
        - center_update() 更新中心
        - cost_function() 算代价
        - plot_clustering() 画图
    """
    centers = init_centers(X, k)
    for epoch in range(1, max_epoch + 1):
        assignment = cluster_assignment(X, centers)
        centers, stop = center_update(assignment, centers)
        if plot:
            plot_clustering(assignment, centers, epoch)
        if stop == 1:
            break
    cost = cost_function(assignment, centers)
    return assignment, cost

## 3. 运行K-Means（k=3）

In [ ]:
k = 3
max_epoch = 200
# 肉眼看着像3簇，先试试k=3
assignment, cost = kmeans(X, k, max_epoch)
print(f"最终代价(Inertia): {cost:.4f}")

## 4. 肘部法则选k

试不同的k，画k-cost曲线，看拐点在哪

In [ ]:
plt.figure(dpi=150)
Cost = []
max_k = 6
# 像胳膊肘，拐点就是最佳k值
for k in range(1, max_k):
    _, cost = kmeans(X, k, max_epoch, False)
    Cost.append(cost)

plt.plot(range(1, max_k), Cost, c='b', marker='*')
plt.xticks(range(1, max_k))
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('肘部法则')
plt.show()

# k=3之后下降变慢了，所以k=3比较合适